In [3]:
import pandas as pd

# --------------------------------------------------
# 1. LOAD MERGED DATASET
# --------------------------------------------------

merged_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/merged_news_prices.csv"

merged = pd.read_csv(merged_path)

print("Merged shape:", merged.shape)
print("Columns:", merged.columns.tolist())
print("\nFirst 5 rows:")
print(merged.head())

# --------------------------------------------------
# 2. BASIC EDA (UNDERSTAND YOUR DATA)
# --------------------------------------------------

print("\nLabel distribution (UpDownLabel):")
print(merged["UpDownLabel"].value_counts(dropna=False))

print("\nTickers in dataset and how many rows each has:")
print(merged["Ticker"].value_counts())

# Check date range
print("\nChecking date range (NewsDate):")
print("NewsDate dtype before:", merged["NewsDate"].dtype)

# Convert NewsDate from string to datetime (no time zone needed here)
merged["NewsDate"] = pd.to_datetime(merged["NewsDate"], errors="coerce")
print("NewsDate dtype after:", merged["NewsDate"].dtype)

print("Earliest NewsDate:", merged["NewsDate"].min())
print("Latest NewsDate:", merged["NewsDate"].max())

# --------------------------------------------------
# 3. CLEAN UP ANY BAD ROWS
# --------------------------------------------------

# Drop rows where we are missing title, label, or date (these are essential)
before_rows = merged.shape[0]
merged = merged.dropna(subset=["title", "UpDownLabel", "NewsDate"])
after_rows = merged.shape[0]

print(f"\nDropped {before_rows - after_rows} rows with missing title/label/date.")
print("New shape:", merged.shape)

# --------------------------------------------------
# 4. TIME-BASED TRAIN / TEST SPLIT
# --------------------------------------------------
# We do NOT shuffle because this is time series data.
# We pick a split date: early data for training, later data for testing.
# For example, use everything before 2019-01-01 for training,
# and 2019 and 2020 for testing.

split_date = pd.Timestamp("2019-01-01")
print("\nUsing split date:", split_date)

train = merged[merged["NewsDate"] < split_date].copy()
test = merged[merged["NewsDate"] >= split_date].copy()

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain date range:")
print("  earliest:", train["NewsDate"].min())
print("  latest:  ", train["NewsDate"].max())

print("\nTest date range:")
print("  earliest:", test["NewsDate"].min())
print("  latest:  ", test["NewsDate"].max())

print("\nTrain label distribution:")
print(train["UpDownLabel"].value_counts())

print("\nTest label distribution:")
print(test["UpDownLabel"].value_counts())

# --------------------------------------------------
# 5. SELECT COLUMNS WE REALLY NEED FOR MODELING
# --------------------------------------------------
# For model training, the most important columns are:
#   - title        (text input)
#   - stock        (ticker from news)
#   - Ticker       (ticker from prices, same as stock)
#   - NewsDate     (date of the news)
#   - ClosePrice   (closing price that day)
#   - NextClose    (next day's closing price)
#   - Return1D     (numeric return)
#   - UpDownLabel  (classification target)

cols_for_model = [
    "title",
    "stock",
    "Ticker",
    "NewsDate",
    "ClosePrice",
    "NextClose",
    "Return1D",
    "UpDownLabel"
]

train_model = train[cols_for_model].copy()
test_model = test[cols_for_model].copy()

print("\nTrain_model sample:")
print(train_model.head())

print("\nTest_model sample:")
print(test_model.head())

# --------------------------------------------------
# 6. SAVE TRAIN AND TEST FILES
# --------------------------------------------------

train_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/train_merged.csv"
test_path = "/Users/hibabelhaj/Desktop/LSTMvsTRANSFORMERS/data/test_merged.csv"

train_model.to_csv(train_path, index=False)
test_model.to_csv(test_path, index=False)

print("\nSaved train file to:", train_path)
print("Saved test file to:", test_path)


Merged shape: (29884, 10)
Columns: ['title', 'date', 'stock', 'NewsDate', 'Ticker', 'TradeDate', 'ClosePrice', 'NextClose', 'Return1D', 'UpDownLabel']

First 5 rows:
                                               title  \
0  BofA Raises Amazon Target On Sales Upside, Acc...   
1  Benzinga's Top Upgrades, Downgrades For June 9...   
2  Wells Fargo Upgrades eBay to Equal-Weight, Ann...   
3  RBC Capital Maintains Sector Perform on eBay, ...   
4  Benchmark Maintains Buy on eBay, Raises Price ...   

                        date stock    NewsDate Ticker   TradeDate  ClosePrice  \
0  2020-06-09 16:35:00+00:00  EBAY  2020-06-09   EBAY  2020-06-09   45.474354   
1  2020-06-09 13:45:00+00:00  EBAY  2020-06-09   EBAY  2020-06-09   45.474354   
2  2020-06-09 10:18:00+00:00  EBAY  2020-06-09   EBAY  2020-06-09   45.474354   
3  2020-06-08 13:03:00+00:00  EBAY  2020-06-08   EBAY  2020-06-08   44.532871   
4  2020-06-05 14:19:00+00:00  EBAY  2020-06-05   EBAY  2020-06-05   44.624279   

   NextClo